### Load the Anthropic API key

In [41]:
from dotenv import load_dotenv

load_dotenv()

True

### Define the anthropic client and model

In [42]:
from anthropic import Anthropic
client = Anthropic()
model = "claude-sonnet-4-6"

### First API call

In [43]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "hello, my name is Juan David"
        }
    ]
)


In [44]:
message.content[0].text

"Hello, Juan David! It's nice to meet you! 😊 How are you doing today? Is there something I can help you with?"

### Let's try to have a conversation

In [45]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[
        {
            "role": "user",
            "content": "What is my name?"
        }
    ]
)

In [46]:
message.content[0].text

"I don't know your name. You haven't shared that information with me. I only know what you tell me within our conversation. What's your name?"

### Let's actually build the conversation by storing the whole list of messages

#### Create helper functions

In [47]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)
    return messages

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)
    return messages

def chat(messages, system=None, temperature=1.0):

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system
    message = client.messages.create(
        **params
    )
    return message.content[0].text

#### Set the system for having a conversation

In [48]:
# Initialize messages list

messages = []

messages = add_user_message(messages, "Hi, my name is Juan David")

response_assistant = chat(messages)

messages = add_assistant_message(messages, response_assistant)

messages = add_user_message(messages, "What is my name?")

response_assistant = chat(messages)

messages = add_assistant_message(messages, response_assistant)

In [49]:
messages

[{'role': 'user', 'content': 'Hi, my name is Juan David'},
 {'role': 'assistant',
  'content': "Hi, Juan David! It's nice to meet you! 😊\n\nHow are you doing? Is there something I can help you with today?"},
 {'role': 'user', 'content': 'What is my name?'},
 {'role': 'assistant',
  'content': 'Your name is **Juan David**! You told me at the beginning of our conversation. 😊'}]

## Create a chatbot (Uncomment if you want to test it)

In [50]:
# # Get input from the user:

# messages = []

# while True:

#     user_message = input("type something")
#     print(f"user: {user_message}")
#     messages = add_user_message(messages, user_message)
#     assistant_message = chat(messages)
#     print(f"assistant: {assistant_message}")
#     messages = add_assistant_message(messages, assistant_message)

## Now test a system prompt

#### Raw claude call

In [51]:
messages = []

messages = add_user_message(
    messages, "how to solve 3x+2=5"
)

assistant_response = chat(messages)

print(assistant_response)

## Solving 3x + 2 = 5

**Goal:** Isolate x by undoing operations in reverse order.

### Steps:

**Step 1: Subtract 2 from both sides**
$$3x + 2 - 2 = 5 - 2$$
$$3x = 3$$

**Step 2: Divide both sides by 3**
$$\frac{3x}{3} = \frac{3}{3}$$
$$x = 1$$

### ✅ Answer: x = 1

### Check:
Plug x = 1 back into the original equation:
- 3(1) + 2 = 5 ✔️


#### Adding a system prompt acting as a tutor

In [52]:
system_prompt = """
You're math tutor, only give hints to the student. Do not
give the answer to the student, guide him.
"""

messages = []

messages = add_user_message(messages, "how to solve 3x+2=5")

assistant_response = chat(messages, system=system_prompt)

print(assistant_response)

Great question! Let's work through this step by step. I'll give you some hints 😊

**Hint 1:** Your goal is to get **x by itself** on one side of the equation.

**Hint 2:** Start by asking yourself — what number is being **added** to 3x? Can you do something to **both sides** of the equation to remove it?

What do you think the first step would be? 🤔


## Exercise

#### Raw claude call

In [53]:
messages = []

messages = add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

assistant_response = chat(messages)

print(assistant_response)

## Check String for Duplicate Characters

Here's a Python function that checks a string for duplicate characters, along with several variations:

```python
def has_duplicates(string: str) -> bool:
    """
    Check if a string contains any duplicate characters.
    
    Args:
        string: The input string to check
        
    Returns:
        True if duplicates exist, False otherwise
    """
    return len(string) != len(set(string))


def get_duplicates(string: str) -> set:
    """
    Return a set of all duplicate characters in a string.
    
    Args:
        string: The input string to check
        
    Returns:
        A set of characters that appear more than once
    """
    seen = set()
    duplicates = set()

    for char in string:
        if char in seen:
            duplicates.add(char)
        else:
            seen.add(char)

    return duplicates


def get_duplicate_counts(string: str) -> dict:
    """
    Return a dictionary of duplicate characters and their counts

#### Now with system prompt

In [54]:
system_prompt = """
You're a staff software engineer that answers as concisely as possible, only give the code to the user, do not explain anything.
"""

messages = []

messages = add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters"
)

assistant_response = chat(messages, system=system_prompt)

print(assistant_response)

```python
def has_duplicate_characters(s: str) -> bool:
    return len(s) != len(set(s))
```


## Let's experiment with temperature

In [55]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=1)

print(answer)

Here's a movie idea:

**A seasoned cartographer discovers that the mysterious, ever-shifting map he inherited from his missing father doesn't chart land — it charts time, and following it means unraveling a conspiracy that could erase entire decades from history.**


In [56]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=0)

print(answer)

Here's a movie idea:

**A retired safecracker with early-onset dementia must break into a high-security vault before his memory fades completely — but he can no longer trust whether the heist is real or something he's imagining.


In [57]:
messages = []

messages = add_user_message(messages, "Generate a one sentence movie idea")

answer = chat(messages, temperature=0)

print(answer)

Here's a movie idea:

**A retired safecracker with early-onset Alzheimer's must pull off one final heist before he forgets the combination he memorized decades ago — the only thing that can prove his son's innocence.**
